In [1]:
import pandas as pd
import json
import requests

In [78]:
import requests
import pandas as pd

BASE = "https://datasette.planning.data.gov.uk"

def _fetch_table(db, table, params=None):
    resp = requests.get(
        f"{BASE}/{db}/{table}.json",
        params={"_shape": "array", "_size": 1000, **(params or {})}
    )
    resp.raise_for_status()
    return resp.json()

def add_resource_info(df):
    # Step 1: fetch org entity ID -> org slug mapping from digital-land
    org_lookup = pd.DataFrame(_fetch_table("digital-land", "organisation"))[["entity", "organisation"]]
    org_lookup["entity"] = pd.to_numeric(org_lookup["entity"], errors="coerce")

    # Step 2: fetch active resources for article-4-direction from performance db
    resource_df = pd.DataFrame(
        _fetch_table("performance", "endpoint_dataset_resource_summary", {"dataset": "article-4-direction"})
    )
    resource_df = resource_df[
        resource_df["resource_end_date"].isna() | (resource_df["resource_end_date"] == "")
    ]
    # If an org has multiple active resources, take the most recently started
    resource_df = (
        resource_df
        .sort_values("resource_start_date", ascending=False)
        .drop_duplicates(subset=["organisation"])
        [["organisation", "resource", "endpoint", "endpoint_url"]]
    )

    # Join everything in pandas
    df = df.copy()
    df["_org_entity"] = pd.to_numeric(df["organisation-entity"], errors="coerce")

    df = df.merge(
        org_lookup.rename(columns={"entity": "_org_entity", "organisation": "_org_slug"}),
        on="_org_entity", how="left"
    )
    df = df.merge(resource_df.rename(columns={"organisation": "_org_slug"}), on="_org_slug", how="left")

    df = df.drop(columns=["organisation", "_org_entity"], errors="ignore")
    df = df.rename(columns={"_org_slug": "organisation"})

    return df


In [ ]:
datasets = ['article-4-direction', 'tree-preservation-orders', 'conservation-area-document']

In [87]:
cad = pd.read_csv("https://files.planning.data.gov.uk/dataset/conservation-area-document.csv")

# Add endpoint and resource columns
cad = add_resource_info(cad)

In [88]:
cad.iloc[0]

dataset                                       conservation-area-document
end-date                                                             NaN
entity                                                           6300000
entry-date                                                    2004-06-01
geojson                                                              NaN
geometry                                                             NaN
name                                                         Mackeye End
organisation-entity                                                  278
point                                                                NaN
prefix                                        conservation-area-document
quality                                                    authoritative
reference                                                            CA5
start-date                                                    1977-07-27
typology                                           

In [89]:
cad.loc[cad.duplicated(subset=['name', 'document-url', 'reference'], keep=False)].sort_values('name')#['organisation-entity'].value_counts()

,dataset,end-date,entity,entry-date,geojson,geometry,name,organisation-entity,point,prefix,...,conservation-area,description,document-type,document-url,documentation-url,notes,organisation,resource,endpoint,endpoint_url
2739,conservation-area-document,NaN,6303514,2008-03-19,NaN,NaN,Annex 4 Boundary,229,NaN,conservation-area-document,...,CONA37,NaN,area-map,https://www.newforestnpa.gov.uk/app/uploads/20...,https://www.newforest.gov.uk/article/3797#CONA37,NFDC/ NFNPA,local-authority:NEW,NaN,NaN,NaN
11743,conservation-area-document,NaN,6314018,2008-03-19,NaN,NaN,Annex 4 Boundary,401,NaN,conservation-area-document,...,CONA37,NaN,area-map,https://www.newforestnpa.gov.uk/app/uploads/20...,https://www.newforestnpa.gov.uk/planning/conse...,NFDC/ NFNPA,national-park-authority:Q72617158,6c2fdf6e4f0e08457196b676330596963ce82046743788...,5bf106eb6a9025d0bc88f0f92373cef3c4a617e9f10777...,https://www.newforestnpa.gov.uk/app/uploads/20...
11744,conservation-area-document,NaN,6314019,2008-03-19,NaN,NaN,Annex 5 Boundary Character Areas,401,NaN,conservation-area-document,...,CONA37,NaN,area-map,https://www.newforestnpa.gov.uk/app/uploads/20...,https://www.newforestnpa.gov.uk/planning/conse...,NFDC/ NFNPA,national-park-authority:Q72617158,6c2fdf6e4f0e08457196b676330596963ce82046743788...,5bf106eb6a9025d0bc88f0f92373cef3c4a617e9f10777...,https://www.newforestnpa.gov.uk/app/uploads/20...
2740,conservation-area-document,NaN,6303515,2008-03-19,NaN,NaN,Annex 5 Boundary Character Areas,229,NaN,conservation-area-document,...,CONA37,NaN,area-map,https://www.newforestnpa.gov.uk/app/uploads/20...,https://www.newforest.gov.uk/article/3797#CONA37,NFDC/ NFNPA,local-authority:NEW,NaN,NaN,NaN
11625,conservation-area-document,NaN,6313900,2000-02-02,NaN,NaN,Appendix 1 Ashlett Creek Conservation Area map,401,NaN,conservation-area-document,...,CONA01,NaN,area-map,https://www.newforest.gov.uk/media/4206/Ashlet...,https://www.newforestnpa.gov.uk/planning/conse...,NFDC/ NFNPA,national-park-authority:Q72617158,6c2fdf6e4f0e08457196b676330596963ce82046743788...,5bf106eb6a9025d0bc88f0f92373cef3c4a617e9f10777...,https://www.newforestnpa.gov.uk/app/uploads/20...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11746,conservation-area-document,NaN,6314021,2008-01-01,NaN,NaN,Western Escarpment Conservation Area,401,NaN,conservation-area-document,...,CONA37,NaN,area-appraisal,https://www.newforestnpa.gov.uk/app/uploads/20...,https://www.newforestnpa.gov.uk/planning/conse...,NFDC/ NFNPA,national-park-authority:Q72617158,6c2fdf6e4f0e08457196b676330596963ce82046743788...,5bf106eb6a9025d0bc88f0f92373cef3c4a617e9f10777...,https://www.newforestnpa.gov.uk/app/uploads/20...
2741,conservation-area-document,NaN,6303516,2008-01-01,NaN,NaN,Western Escarpment Conservation Area Action Pl...,229,NaN,conservation-area-document,...,CONA37,NaN,NaN,https://www.newforestnpa.gov.uk/app/uploads/20...,https://www.newforest.gov.uk/article/3797#CONA37,NFDC/ NFNPA,local-authority:NEW,NaN,NaN,NaN
11745,conservation-area-document,NaN,6314020,2008-01-01,NaN,NaN,Western Escarpment Conservation Area Action Pl...,401,NaN,conservation-area-document,...,CONA37,NaN,management-plan,https://www.newforestnpa.gov.uk/app/uploads/20...,https://www.newforestnpa.gov.uk/planning/conse...,NFDC/ NFNPA,national-park-authority:Q72617158,6c2fdf6e4f0e08457196b676330596963ce82046743788...,5bf106eb6a9025d0bc88f0f92373cef3c4a617e9f10777...,https://www.newforestnpa.gov.uk/app/uploads/20...
11741,conservation-area-document,NaN,6314016,2009-03-01,NaN,NaN,Western Escarpment Conservation Area Character...,401,NaN,conservation-area-document,...,CONA37,NaN,area-appraisal,https://www.newforestnpa.gov.uk/app/uploads/20...,https://www.newforestnpa.gov.uk/planning/conse...,NFDC/ NFNPA,national-park-authority:Q72617158,6c2fdf6e4f0e08457196b676330596963ce82046743788...,5bf106eb6a9025d0bc88f0f92373cef3c4a617e9f10777...,https://www.newforestnpa.gov.uk/app/uploads/20...


---

## Article 4 Direction

In [35]:
a4d = pd.read_csv("https://files.planning.data.gov.uk/dataset/article-4-direction.csv")

In [ ]:
# Add endpoint and resource columns
a4d = add_resource_info(a4d)

In [95]:
dupe_cols = ['name'] #, 'document-url', 'start-date', 'description'

(
    a4d.loc[a4d.duplicated(subset=dupe_cols, keep=False)]
    .groupby(dupe_cols, dropna=False)
    .filter(lambda g: g['endpoint_url'].nunique() > 1 or g['resource'].nunique() > 1)
)


,dataset,end-date,entity,entry-date,geojson,geometry,name,organisation-entity,point,prefix,...,start-date,typology,description,document-url,documentation-url,notes,organisation,resource,endpoint,endpoint_url
2040,article-4-direction,NaN,6102253,2026-01-20,NaN,NaN,Green Lane,383,NaN,article-4-direction,...,2016-07-27,legal-instrument,NaN,https://www.westoxon.gov.uk/media/zvqap2r4/art...,https://www.westoxon.gov.uk/planning-and-build...,NaN,local-authority:WOX,98cd03f22297a735e5411d19c45641148253bda0493ed6...,42131d84543ad864ed9d306eedf3efa33003302d7c4398...,https://www.westoxon.gov.uk/media/vgtiuf5r/wod...
2102,article-4-direction,NaN,6102315,2026-01-24,NaN,NaN,Green Lane,299,NaN,article-4-direction,...,2006-07-12,legal-instrument,NaN,https://smbc-opendata.s3.eu-west-1.amazonaws.c...,https://www.stockport.gov.uk/directories/entry...,NaN,local-authority:SKP,96afb8aac4f678a841bc034cb8e2bdfb5071bce0136604...,b03a22ccc8e1b3e98f81fd528407b79a18fe72b786e37c...,https://smbc-opendata.s3.eu-west-1.amazonaws.c...
2159,article-4-direction,NaN,6102372,2026-04-25,NaN,NaN,Green Lane,299,NaN,article-4-direction,...,2006-07-12,legal-instrument,NaN,https://smbc-opendata.s3.eu-west-1.amazonaws.c...,https://www.stockport.gov.uk/directories/entry...,NaN,local-authority:SKP,96afb8aac4f678a841bc034cb8e2bdfb5071bce0136604...,b03a22ccc8e1b3e98f81fd528407b79a18fe72b786e37c...,https://smbc-opendata.s3.eu-west-1.amazonaws.c...


In [ ]:
a4d_dupes = a4d.loc[a4d.duplicated(subset=['name', 'document-url', 'start-date', 'description'], keep=False)].sort_values('document-url')
a4d_dupes

,dataset,end-date,entity,entry-date,geojson,geometry,name,organisation-entity,point,prefix,...,start-date,typology,description,document-url,documentation-url,notes,organisation,resource,endpoint,endpoint_url
2488,article-4-direction,NaN,6102701,1998-01-23,NaN,NaN,Greywell,161,NaN,article-4-direction,...,1998-01-23,legal-instrument,NaN,http://52.17.42.98/Hartdata/article4/29article...,https://www.hart.gov.uk/planning-and-building-...,NaN,local-authority:HAT,86477bba7812bfe50e01cdbfc3014d11ae6a475b4e78a0...,45ee7ec4dfca03ef566251a320261a19f7b2f2930def06...,https://www.hart.gov.uk/sites/default/files/20...
2464,article-4-direction,NaN,6102677,1998-01-23,NaN,NaN,Greywell,161,NaN,article-4-direction,...,1998-01-23,legal-instrument,NaN,http://52.17.42.98/Hartdata/article4/29article...,https://www.hart.gov.uk/planning-and-building-...,NaN,local-authority:HAT,86477bba7812bfe50e01cdbfc3014d11ae6a475b4e78a0...,45ee7ec4dfca03ef566251a320261a19f7b2f2930def06...,https://www.hart.gov.uk/sites/default/files/20...
290,article-4-direction,NaN,6100432,2025-03-21,NaN,NaN,Broads Authority Conservation Area,61,NaN,article-4-direction,...,2021-01-01,legal-instrument,Removes Permitted Development right for Solar ...,http://www.broads-authority.gov.uk/planning/ot...,http://www.broads-authority.gov.uk/planning/ot...,NaN,local-authority:BRO,fc9f13a2a44935ad973b4e8dae9cba0b354d0cb2dfd6ca...,8250a923ab931a7929bee8fb7723b95621fd90219f7db5...,https://www.southnorfolkandbroadland.gov.uk/as...
271,article-4-direction,NaN,6100413,2025-03-21,NaN,NaN,Broads Authority,61,NaN,article-4-direction,...,1972-01-01,legal-instrument,Restricts the use of the land covered by water...,http://www.broads-authority.gov.uk/planning/ot...,http://www.broads-authority.gov.uk/planning/ot...,NaN,local-authority:BRO,fc9f13a2a44935ad973b4e8dae9cba0b354d0cb2dfd6ca...,8250a923ab931a7929bee8fb7723b95621fd90219f7db5...,https://www.southnorfolkandbroadland.gov.uk/as...
273,article-4-direction,NaN,6100415,2025-03-21,NaN,NaN,Broads Authority,61,NaN,article-4-direction,...,1972-01-01,legal-instrument,Restricts the use of the land covered by water...,http://www.broads-authority.gov.uk/planning/ot...,http://www.broads-authority.gov.uk/planning/ot...,NaN,local-authority:BRO,fc9f13a2a44935ad973b4e8dae9cba0b354d0cb2dfd6ca...,8250a923ab931a7929bee8fb7723b95621fd90219f7db5...,https://www.southnorfolkandbroadland.gov.uk/as...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
257,article-4-direction,NaN,6100399,1985-01-01,NaN,NaN,Caldy/Thurstaston. Article 4 Direction 1985,384,NaN,article-4-direction,...,1985-01-01,legal-instrument,Caldy/Thurstaston,https://www.wirral.gov.uk/files/caldy-thurstas...,https://www.wirral.gov.uk/planning-and-buildin...,Article 4 Direction,local-authority:WRL,fd70c4120e4542226ab1f5ac79171d60e80c96c5785e4a...,ca4bb4ce3d76cc57183c2de03dfddb6f3e45b451c658b0...,https://www.wirral.gov.uk/files/article-4-dire...
256,article-4-direction,NaN,6100398,1985-01-01,NaN,NaN,Caldy/Thurstaston. Article 4 Direction 1985,384,NaN,article-4-direction,...,1985-01-01,legal-instrument,Caldy/Thurstaston,https://www.wirral.gov.uk/files/caldy-thurstas...,https://www.wirral.gov.uk/planning-and-buildin...,Article 4 Direction,local-authority:WRL,fd70c4120e4542226ab1f5ac79171d60e80c96c5785e4a...,ca4bb4ce3d76cc57183c2de03dfddb6f3e45b451c658b0...,https://www.wirral.gov.uk/files/article-4-dire...
255,article-4-direction,NaN,6100397,1985-01-01,NaN,NaN,Caldy/Thurstaston. Article 4 Direction 1985,384,NaN,article-4-direction,...,1985-01-01,legal-instrument,Caldy/Thurstaston,https://www.wirral.gov.uk/files/caldy-thurstas...,https://www.wirral.gov.uk/planning-and-buildin...,Article 4 Direction,local-authority:WRL,fd70c4120e4542226ab1f5ac79171d60e80c96c5785e4a...,ca4bb4ce3d76cc57183c2de03dfddb6f3e45b451c658b0...,https://www.wirral.gov.uk/files/article-4-dire...
2319,article-4-direction,NaN,6102532,1993-04-02,NaN,NaN,Land at North Downs east of Hollingbourne,205,NaN,article-

There are a potential of 268 dupes across the A4D, if finding duplicated records by:
- name
- document-url
- start-date
- description

In [ ]:
a4d.loc[a4d['name'] == 'Clarendon Place']['document-url'].values.tolist()


['https://docs.maidstone.gov.uk/planning/article-4/offices-to-residential/Article-4-Clarendon-Place.pdf',
 'https://docs.maidstone.gov.uk/planning/article-4/offices-to-residential/Article-4-Clarendon-Place.pdf']

In [65]:
a4d.loc[a4d['name'] == 'Greywell']['document-url'].values.tolist()

['http://52.17.42.98/Hartdata/article4/29article4s_23-01-1998.pdf',
 'http://52.17.42.98/Hartdata/article4/29article4s_23-01-1998.pdf']

In [71]:
a4d.loc[a4d['name'] == 'Clarendon Place']#['documentation-url'].values.tolist()

,dataset,end-date,entity,entry-date,geojson,geometry,name,organisation-entity,point,prefix,quality,reference,start-date,typology,description,document-url,documentation-url,notes,organisation
2358,article-4-direction,NaN,6102571,2019-12-13,NaN,NaN,Clarendon Place,205,NaN,article-4-direction,authoritative,A4D68,2019-11-07,legal-instrument,Article 4(1) Direction of the Town and Country...,https://docs.maidstone.gov.uk/planning/article...,https://maidstone.gov.uk/home/primary-services...,NaN,NaN
2369,article-4-direction,NaN,6102582,2022-02-18,NaN,NaN,Clarendon Place,205,NaN,article-4-direction,authoritative,A4D79,2022-02-17,legal-instrument,Article 4(1) Non-Immediate Direction of the To...,https://docs.maidstone.gov.uk/planning/article...,https://maidstone.gov.uk/home/primary-services...,NaN,NaN


In [57]:
a4da = pd.read_csv("https://files.planning.data.gov.uk/dataset/article-4-direction-area.csv")

In [67]:
a4da.loc[a4da['name'] == 'Clarendon Place']

,dataset,end-date,entity,entry-date,geojson,geometry,name,organisation-entity,point,prefix,...,reference,start-date,typology,address-texts,article-4-direction,description,notes,organisation,permitted-development-rights,uprns
4339,article-4-direction-area,NaN,7010007500,2019-12-13,NaN,"MULTIPOLYGON (((0.528359 51.273471,0.528874 51...",Clarendon Place,205,POINT (0.52884 51.273509),article-4-direction-area,...,A4D68,2019-11-07,geography,NaN,A4D68,NaN,NaN,NaN,CodeUnknown,NaN
4350,article-4-direction-area,NaN,7010007511,2022-02-18,NaN,"MULTIPOLYGON (((0.528359 51.273471,0.528874 51...",Clarendon Place,205,POINT (0.52884 51.273509),article-4-direction-area,...,A4D79,2022-02-17,geography,NaN,A4D79,NaN,NaN,NaN,CodeUnknown,NaN
